In [1]:
import pandas as pd
import json

In [2]:
business = pd.read_json("yelp_academic_dataset_business.json", lines = True)

business = business[
    ['business_id', 'name', 'city', 'state', 'stars', 'review_count', 'categories']
]

business = business.dropna(subset=['categories'])

business = business[business['categories'].str.contains('Restaurants', na=False)]

business['categories'] = business['categories'].str.split(', ')


In [3]:
def extract_cuisine(category_list):
    category_list = [c.lower() for c in category_list]

    def contains_any(keywords):
        return any(any(k in c for k in keywords) for c in category_list)
    if contains_any(['american (traditional)', 'american (new)', 'burgers']):
        return 'American'
    if contains_any(['indian']):
        return 'Indian'
    if contains_any(['chinese', 'szechuan', 'cantonese']):
        return 'Chinese'
    if contains_any(['mexican']):
        return 'Mexican'
    if contains_any(['italian']):
        return 'Italian'
    if contains_any(['japanese', 'sushi', 'ramen']):
        return 'Japanese'
    if contains_any(['thai']):
        return 'Thai'
    if contains_any(['korean']):
        return 'Korean'
    if contains_any(['greek']):
        return 'Greek'
    if contains_any(['french']):
        return 'French'

    return None
business['cuisine'] = business['categories'].apply(extract_cuisine)

business_filtered = business[business['cuisine'].notna()]

print(business['cuisine'].value_counts())

cuisine
American    15949
Mexican      4112
Italian      3910
Chinese      3031
Japanese     1903
Indian        809
Greek         575
Thai          573
French        286
Korean        284
Name: count, dtype: int64


In [4]:
chains_to_group = [
    "Taco Bell", "Pizza Hut", "Chipotle Mexican Grill", "Panda Express",
    "Jack in the Box", "QDOBA Mexican Eats", "Moe's Southwest Grill",
    "Olive Garden Italian Restaurant", "Domino's Pizza", "Jet's Pizza",
    "Carrabba's Italian Grill", "Boston Pizza", "Tijuana Flats",
    "Zoes Kitchen", "Noodles & Company", "Fazoli's",
    "Romano's Macaroni Grill", "P.F. Chang's",
    "Papa John's Pizza", "Pei Wei"
]

chains_df = business_filtered[business_filtered['name'].isin(chains_to_group)]
non_chains_df = business_filtered[~business_filtered['name'].isin(chains_to_group)]
chains_grouped = chains_df.drop_duplicates(subset=['name','state'])

business_filtered = pd.concat([non_chains_df, chains_grouped])

In [5]:
business_filtered.to_csv("clean_business.csv", index=False)

In [6]:
business_ids = set(business_filtered['business_id'])

filtered_reviews = []

with open("yelp_academic_dataset_review.json", 'r') as f:
    for i, line in enumerate(f):
        review = json.loads(line)
        
        review_date = review['date'][:10]
        
        if review_date >= "2021-01-01":
            if review['business_id'] in business_ids:
                filtered_reviews.append({
                    'business_id': review['business_id'],
                    'text': review['text'],
                    'stars': review['stars'],
                    'date': review['date'],
                    'user_id': review['user_id']
                })
        
        if len(filtered_reviews) >= 100000:
            break
        
        if i % 500000 == 0:
            print(f"Processed {i} lines...")

Processed 0 lines...
Processed 500000 lines...
Processed 1000000 lines...
Processed 1500000 lines...
Processed 2000000 lines...
Processed 2500000 lines...


In [7]:
reviews = pd.DataFrame(filtered_reviews)
reviews['date'] = pd.to_datetime(reviews['date'])
reviews['text'] = reviews['text'].str.lower()
reviews = reviews[reviews['text'].str.len() > 20]

In [8]:
reviews.to_csv("clean_reviews.csv", index=False)

In [9]:
final_df = reviews.merge(business_filtered, on='business_id')
print(final_df.shape)
print(final_df['cuisine'].value_counts())

(100000, 12)
cuisine
American    53487
Mexican     12773
Italian     11705
Japanese     8028
Chinese      6013
Indian       2296
Thai         2087
Greek        1601
Korean       1082
French        928
Name: count, dtype: int64


In [10]:
final_df['text'] = final_df['text'].str.lower()

final_df = final_df[final_df['text'].str.len() > 20]

final_df = final_df.rename(columns={
    'stars_x': 'individual_stars',
    'stars_y': 'total_business_stars'
})

In [11]:
final_df.to_csv("final_yelp_dataset.csv", index=False)

In [12]:
user_ids = set(final_df['user_id'])
filtered_users = []

with open("yelp_academic_dataset_user.json", 'r') as f:
    for i, line in enumerate(f):
        user = json.loads(line)
        
        if user['user_id'] in user_ids:
            filtered_users.append({
                'user_id': user['user_id'],
                'review_count': user['review_count'],
                'elite': user['elite'],
                'average_stars': user['average_stars']
            })
        
        if i % 100000 == 0:
            print(f"Processed {i} users...")

Processed 0 users...
Processed 100000 users...
Processed 200000 users...
Processed 300000 users...
Processed 400000 users...
Processed 500000 users...
Processed 600000 users...
Processed 700000 users...
Processed 800000 users...
Processed 900000 users...
Processed 1000000 users...
Processed 1100000 users...
Processed 1200000 users...
Processed 1300000 users...
Processed 1400000 users...
Processed 1500000 users...
Processed 1600000 users...
Processed 1700000 users...
Processed 1800000 users...
Processed 1900000 users...


In [13]:
users = pd.DataFrame(filtered_users)

users['is_elite'] = users['elite'].apply(lambda x: 0 if x == 'None' else 1)

users = users.drop(columns=['elite']) 


In [14]:
users.to_csv("clean_users.csv", index=False)

In [16]:
final_df.groupby('cuisine')['business_id'].nunique()

cuisine
American    3774
Chinese      758
French        49
Greek        115
Indian       159
Italian      851
Japanese     466
Korean        70
Mexican      904
Thai         142
Name: business_id, dtype: int64